# 🎨 ComfyUI Colab — WAI-illustrious + YOLO + SAM

**Thứ tự chạy:** Cell **1 → 1B → 2 → 3**  (đã tách vì Cell 1 gộp YOLO hay bị “treo” im lặng)

1. Runtime → Change runtime type → **T4 GPU** → Save
2. Cell 1 xong khi thấy `✅ Xong Cell 1` (~2–3 phút, **sẽ có chữ in từng bước**)
3. Cell 1B cài Impact Pack (~30–60 giây)
4. Cell 2 dán Civitai key, tải WAI + YOLO + SAM
5. Cell 3 lấy link → dán tab mới → kéo `WAI_ChiTietNho.json` → **R** → chọn model

Nếu Cell 1 đứng ở `Mounting Drive` quá 1 phút: cửa sổ xin quyền bị chặn → cho phép popup, hoặc tick **Bỏ qua Drive** rồi chạy lại.

In [ ]:
# ===== CELL 1: Drive + ComfyUI (KHÔNG cài YOLO ở đây — tránh treo pip) =====
BO_QUA_DRIVE = False  # @param {type:"boolean"}

import os, time, sys
def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)
    sys.stdout.flush()

log('GPU:')
!nvidia-smi --query-gpu=name,memory.total --format=csv

USE_DRIVE = False
if not BO_QUA_DRIVE:
    log('Kết nối Drive — nếu đứng >60s: tick BO_QUA_DRIVE, Runtime → Interrupt, chạy lại.')
    try:
        from google.colab import drive
        drive.mount('/content/drive')  # KHÔNG force_remount (dễ treo)
        USE_DRIVE = True
        log('Drive OK')
    except Exception as e:
        log(f'Drive lỗi ({e}) → ổ tạm')
else:
    log('Bỏ qua Drive (ổ tạm, model mất khi ngắt phiên)')

ROOT = '/content/drive/MyDrive/AI_Models' if USE_DRIVE else '/content/AI_Models'
for p in [f'{ROOT}/checkpoints', f'{ROOT}/ultralytics/bbox', f'{ROOT}/sams']:
    os.makedirs(p, exist_ok=True)
CKPT_DIR, YOLO_DIR, SAM_DIR = f'{ROOT}/checkpoints', f'{ROOT}/ultralytics/bbox', f'{ROOT}/sams'
open('/content/ckpt_dir.txt','w').write(CKPT_DIR)
open('/content/yolo_dir.txt','w').write(YOLO_DIR)
open('/content/sam_dir.txt','w').write(SAM_DIR)
open('/content/root_dir.txt','w').write(ROOT)

os.chdir('/content')
if os.path.isdir('/content/ComfyUI') and os.path.isfile('/content/ComfyUI/main.py'):
    log('ComfyUI đã có — không clone lại')
else:
    log('Clone ComfyUI (~1–2 phút, có % là chưa chết)...')
    !git clone --progress --depth 1 https://github.com/comfyanonymous/ComfyUI /content/ComfyUI

os.chdir('/content/ComfyUI')
log('Cài thư viện ComfyUI (giữ PyTorch GPU của Colab, ~1 phút)...')
!grep -viE '^(torch|torchvision|torchaudio)([=<>!~ ]|$)' requirements.txt > /content/req_notorch.txt
!pip install -r /content/req_notorch.txt

import torch
assert torch.cuda.is_available(), '❌ Không thấy GPU. Runtime → Change runtime type → T4 GPU → Restart session → Cell 1'
log(f'PyTorch {torch.__version__} | GPU {torch.cuda.get_device_name(0)}')

log('✅ Xong Cell 1 — chạy tiếp Cell 1B')

In [ ]:
# ===== CELL 1B: Impact Pack + YOLO/SAM (nhẹ, không đè OpenCV/Torch) =====
import os, time, sys, subprocess
def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)
    sys.stdout.flush()

ROOT = open('/content/root_dir.txt').read().strip()
CKPT_DIR = open('/content/ckpt_dir.txt').read().strip()
YOLO_DIR = open('/content/yolo_dir.txt').read().strip()
SAM_DIR  = open('/content/sam_dir.txt').read().strip()
CN = '/content/ComfyUI/custom_nodes'
os.makedirs(CN, exist_ok=True)
os.chdir(CN)

def clone(url, folder):
    path = os.path.join(CN, folder)
    if os.path.isdir(path) and os.listdir(path):
        log(f'{folder} đã có — bỏ qua clone')
        return
    log(f'Clone {folder}...')
    r = subprocess.run(['git','clone','--progress','--depth','1', url, path])
    if r.returncode != 0:
        raise RuntimeError(f'Clone {folder} thất bại')

clone('https://github.com/ltdrdata/ComfyUI-Impact-Pack.git', 'ComfyUI-Impact-Pack')
clone('https://github.com/ltdrdata/ComfyUI-Impact-Subpack.git', 'ComfyUI-Impact-Subpack')

log('pip nhẹ: piexif dill segment-anything + ultralytics --no-deps (không cài lại opencv/torch)')
!pip install piexif dill segment-anything
!pip install ultralytics --no-deps

log('Symlink checkpoints / YOLO / SAM')
pairs = [
    ('/content/ComfyUI/models/checkpoints', CKPT_DIR),
    ('/content/ComfyUI/models/ultralytics', f'{ROOT}/ultralytics'),
    ('/content/ComfyUI/models/sams', SAM_DIR),
]
for path, dest in pairs:
    os.makedirs(dest, exist_ok=True)
    if os.path.islink(path) or os.path.exists(path):
        if os.path.islink(path) or os.path.isdir(path):
            !rm -rf "{path}"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    os.symlink(dest, path)
    log(f'  {path} → {dest}')

print()
!ls /content/ComfyUI/custom_nodes | grep -i impact
log('✅ Xong Cell 1B — chạy Cell 2 (tải file .pt / .safetensors)')

In [ ]:
# ===== CELL 2: Tải WAI + YOLO mặt/tay + SAM =====
CIVITAI_KEY = ""  # @param {type:"string"}

import os, time, sys
def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True); sys.stdout.flush()

CKPT_DIR = open('/content/ckpt_dir.txt').read().strip()
YOLO_DIR = open('/content/yolo_dir.txt').read().strip()
SAM_DIR  = open('/content/sam_dir.txt').read().strip()
os.makedirs(YOLO_DIR, exist_ok=True)
os.makedirs(SAM_DIR, exist_ok=True)
log(f'Checkpoint {CKPT_DIR}')
log(f'YOLO {YOLO_DIR}')
log(f'SAM  {SAM_DIR}')

def download(url, path, min_bytes, label):
    if os.path.exists(path) and os.path.getsize(path) > min_bytes:
        log(f'✅ {label} đã có ({os.path.getsize(path)/1e6:.0f} MB)')
        return
    log(f'⬇️  {label}')
    !wget --show-progress -c -O "{path}" "{url}"
    size = os.path.getsize(path) if os.path.exists(path) else 0
    if size < min_bytes:
        if os.path.exists(path): os.remove(path)
        raise RuntimeError(f'❌ {label} thất bại ({size} byte)')
    log(f'✅ {label} ({size/1e6:.0f} MB)')

WAI = f'{CKPT_DIR}/WAI-illustrious.safetensors'
if os.path.exists(WAI) and os.path.getsize(WAI) > 6_000_000_000:
    log('✅ WAI-illustrious đã có')
else:
    assert CIVITAI_KEY.strip(), '❌ Dán Civitai API key vào ô CIVITAI_KEY'
    download(
        f'https://civitai.com/api/download/models/2514310?token={CIVITAI_KEY.strip()}',
        WAI, 6_000_000_000, 'WAI-illustrious')

download('https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8m.pt',
         f'{YOLO_DIR}/face_yolov8m.pt', 20_000_000, 'YOLO mặt')
download('https://huggingface.co/Bingsu/adetailer/resolve/main/hand_yolov8s.pt',
         f'{YOLO_DIR}/hand_yolov8s.pt', 8_000_000, 'YOLO tay')
download('https://huggingface.co/Claquasse/foot_anime_yolo/resolve/main/foot_anime_yolo11m_v3.pt',
         f'{YOLO_DIR}/foot_anime_yolo11m_v3.pt', 15_000_000, 'YOLO chân anime v3')

SAM = f'{SAM_DIR}/sam_vit_b_01ec64.pth'
try:
    download('https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth',
             SAM, 300_000_000, 'SAM ViT-B')
except Exception as e:
    log(f'Facebook lỗi ({e}), thử HF')
    download('https://huggingface.co/ybelkada/segment-anything/resolve/main/checkpoints/sam_vit_b_01ec64.pth',
             SAM, 300_000_000, 'SAM ViT-B (HF)')

!ls -lh {CKPT_DIR}
!ls -lh {YOLO_DIR}
!ls -lh {SAM_DIR}
log('✅ Cell 2 xong — chạy Cell 3')

In [ ]:
# @title ⚙️ CELL 3 — Khởi chạy ComfyUI (đầy đủ tùy chỉnh)
# Điều kiện: Cell 1 + 1B + 2 đã xong. Điền form bên phải / bên dưới rồi ▶ chạy.
# Cell xong = ComfyUI + tunnel VẪN CHẠY NỀN. Chạy lại Cell 3 = restart server.

# ----- Đường dẫn & mạng -----
LISTEN = "0.0.0.0"  # @param {type:"string"}
PORT = 8188  # @param {type:"integer"}
TUNNEL = "cloudflared http2 (mạng VN)"  # @param ["cloudflared http2 (mạng VN)", "cloudflared quic", "không tunnel (chỉ Colab local)", "bỏ qua tunnel — ComfyUI only"]
CORS = True  # @param {type:"boolean"}
CORS_ORIGIN = "*"  # @param {type:"string"}

# ----- VRAM / GPU (T4 16GB: để Mặc định hoặc normalvram) -----
VRAM = "Mặc định (T4)"  # @param ["Mặc định (T4)", "highvram", "normalvram", "lowvram", "novram", "cpu"]
RESERVE_VRAM_GB = 0.6  # @param {type:"slider", min:0.0, max:4.0, step:0.1}
CUDA_DEVICE = 0  # @param {type:"integer"}
CUDA_MALLOC = True  # @param {type:"boolean"}
DISABLE_SMART_MEMORY = False  # @param {type:"boolean"}

# ----- Độ chính xác -----
FORCE_PREC = "fp16"  # @param ["fp16", "fp32", "không ép"]
UNET_PREC = "Mặc định"  # @param ["Mặc định", "fp16-unet", "bf16-unet", "fp8_e4m3fn-unet"]
VAE_PREC = "fp16-vae"  # @param ["Mặc định", "fp16-vae", "fp32-vae", "bf16-vae", "cpu-vae"]

# ----- Attention / preview -----
ATTENTION = "pytorch"  # @param ["pytorch", "split", "quad", "sage", "flash", "xformers (nếu có)", "Mặc định"]
PREVIEW = "auto"  # @param ["auto", "latent2rgb", "taesd", "none"]
FAST = False  # @param {type:"boolean"}
DISABLE_XFORMERS = True  # @param {type:"boolean"}

# ----- Thư mục -----
OUTPUT_DIR = "/content/ComfyUI/output"  # @param {type:"string"}
INPUT_DIR = "/content/ComfyUI/input"  # @param {type:"string"}
TEMP_DIR = "/content/ComfyUI/temp"  # @param {type:"string"}

# ----- Hành vi cell -----
KILL_OLD = True  # @param {type:"boolean"}
CHECK_MODELS = True  # @param {type:"boolean"}
WAIT_SECONDS = 180  # @param {type:"integer"}
EXTRA_ARGS = ""  # @param {type:"string"}
VERBOSE_LOG = True  # @param {type:"boolean"}

# =============================================================================
import os, sys, time, socket, re, subprocess, shutil, urllib.request

def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)
    sys.stdout.flush()

COMFY = '/content/ComfyUI'
assert os.path.isfile(f'{COMFY}/main.py'), '❌ Chưa có ComfyUI — chạy Cell 1 trước'

# ----- kiểm tra model (tùy chọn) -----
if CHECK_MODELS:
    log('Kiểm tra model / YOLO / SAM / Impact...')
    def ls(p):
        try:
            return os.listdir(p)
        except Exception:
            return []
    ckpt, yolo, sams = f'{COMFY}/models/checkpoints', f'{COMFY}/models/ultralytics/bbox', f'{COMFY}/models/sams'
    print('  Impact Pack :', os.path.isdir(f'{COMFY}/custom_nodes/ComfyUI-Impact-Pack'))
    print('  Impact Sub  :', os.path.isdir(f'{COMFY}/custom_nodes/ComfyUI-Impact-Subpack'))
    print('  checkpoints :', ls(ckpt))
    print('  yolo bbox   :', ls(yolo))
    print('  sams        :', ls(sams))
    miss = []
    if not any('WAI' in x.upper() or 'wai' in x for x in ls(ckpt)):
        miss.append('WAI-illustrious.safetensors (Cell 2)')
    if not any('face' in x for x in ls(yolo)):
        miss.append('face_yolov8m.pt')
    if not any('hand' in x for x in ls(yolo)):
        miss.append('hand_yolov8s.pt')
    if not any('foot' in x for x in ls(yolo)):
        miss.append('foot_anime_yolo11m_v3.pt (chân)')
    if not any(x.endswith('.pth') for x in ls(sams)):
        miss.append('sam_vit_b_01ec64.pth')
    if miss:
        log('⚠️ Thiếu: ' + ' | '.join(miss))
    else:
        log('✅ Đủ checkpoint + YOLO mặt/tay/chân + SAM')

# ----- dọn process cũ -----
if KILL_OLD:
    log('Dừng ComfyUI / cloudflared cũ...')
    os.system('pkill -f "python main.py" >/dev/null 2>&1 || true')
    os.system('pkill -f cloudflared >/dev/null 2>&1 || true')
    time.sleep(2)

# ----- cloudflared -----
need_tunnel = TUNNEL.startswith('cloudflared')
if need_tunnel and not shutil.which('cloudflared') and not os.path.exists('/usr/local/bin/cloudflared'):
    log('Cài cloudflared...')
    os.system('wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /tmp/cloudflared.deb')
    os.system('dpkg -i /tmp/cloudflared.deb >/dev/null 2>&1')

# ----- ghép argv ComfyUI từ form -----
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(TEMP_DIR, exist_ok=True)

cmd = [sys.executable, 'main.py',
       '--listen', str(LISTEN),
       '--port', str(int(PORT)),
       '--preview-method', PREVIEW,
       '--output-directory', OUTPUT_DIR,
       '--input-directory', INPUT_DIR,
       '--temp-directory', TEMP_DIR,
       '--cuda-device', str(int(CUDA_DEVICE))]
if CORS:
    cmd += ['--enable-cors-header', CORS_ORIGIN or '*']
vram_map = {
    'highvram': '--highvram', 'normalvram': '--normalvram',
    'lowvram': '--lowvram', 'novram': '--novram', 'cpu': '--cpu',
}
if VRAM in vram_map:
    cmd.append(vram_map[VRAM])
if RESERVE_VRAM_GB and RESERVE_VRAM_GB > 0:
    cmd += ['--reserve-vram', str(float(RESERVE_VRAM_GB))]
if CUDA_MALLOC:
    cmd.append('--cuda-malloc')
else:
    cmd.append('--disable-cuda-malloc')
if DISABLE_SMART_MEMORY:
    cmd.append('--disable-smart-memory')
if FORCE_PREC == 'fp16':
    cmd.append('--force-fp16')
elif FORCE_PREC == 'fp32':
    cmd.append('--force-fp32')
unet_map = {'fp16-unet': '--fp16-unet', 'bf16-unet': '--bf16-unet', 'fp8_e4m3fn-unet': '--fp8_e4m3fn-unet'}
if UNET_PREC in unet_map:
    cmd.append(unet_map[UNET_PREC])
vae_map = {'fp16-vae': '--fp16-vae', 'fp32-vae': '--fp32-vae', 'bf16-vae': '--bf16-vae', 'cpu-vae': '--cpu-vae'}
if VAE_PREC in vae_map:
    cmd.append(vae_map[VAE_PREC])
att_map = {
    'pytorch': '--use-pytorch-cross-attention',
    'split': '--use-split-cross-attention',
    'quad': '--use-quad-cross-attention',
    'sage': '--use-sage-attention',
    'flash': '--use-flash-attention',
}
if ATTENTION in att_map:
    cmd.append(att_map[ATTENTION])
if DISABLE_XFORMERS or ATTENTION != 'xformers (nếu có)':
    cmd.append('--disable-xformers')
if FAST:
    cmd.append('--fast')
if VERBOSE_LOG:
    cmd += ['--verbose', 'INFO']
if EXTRA_ARGS.strip():
    cmd += EXTRA_ARGS.strip().split()

log('Lệnh ComfyUI:')
print(' ', ' '.join(cmd), flush=True)

log_path = '/content/comfyui.log'
comfy_log = open(log_path, 'w')
comfy = subprocess.Popen(cmd, stdout=comfy_log, stderr=subprocess.STDOUT, cwd=COMFY)
open('/content/comfy.pid', 'w').write(str(comfy.pid))
log(f'pid={comfy.pid} — đợi cổng {PORT} (tối đa {WAIT_SECONDS}s)...')

def port_open():
    try:
        with socket.create_connection(('127.0.0.1', int(PORT)), timeout=1):
            return True
    except OSError:
        return False

ok_port = False
for i in range(int(WAIT_SECONDS)):
    time.sleep(1)
    if comfy.poll() is not None:
        print('\n----- /content/comfyui.log -----')
        os.system('tail -60 /content/comfyui.log')
        raise RuntimeError('❌ ComfyUI chết khi khởi động. Đổi VRAM=lowvram hoặc tắt FAST rồi chạy lại.')
    if i in (15, 30, 60, 90, 120):
        log(f'  ... {i}s')
        if VERBOSE_LOG:
            os.system('tail -4 /content/comfyui.log')
    if port_open():
        ok_port = True
        break
if not ok_port:
    os.system('tail -60 /content/comfyui.log')
    raise RuntimeError(f'❌ Quá {WAIT_SECONDS}s chưa mở cổng {PORT}')
log(f'✅ ComfyUI listen {LISTEN}:{PORT}')
try:
    with urllib.request.urlopen(f'http://127.0.0.1:{int(PORT)}/system_stats', timeout=10) as r:
        log(f'   /system_stats HTTP {r.status}')
except Exception as e:
    log(f'   /system_stats: {e}')

# ----- tunnel -----
url = None
if need_tunnel:
    proto = 'http2' if 'http2' in TUNNEL else 'quic'
    cf_log_path = '/content/cloudflared.log'
    cf_log = open(cf_log_path, 'w')
    cf_cmd = ['cloudflared', 'tunnel',
              '--url', f'http://127.0.0.1:{int(PORT)}',
              '--http-host-header', f'127.0.0.1:{int(PORT)}',
              '--protocol', proto]
    log('Tunnel: ' + ' '.join(cf_cmd))
    cf = subprocess.Popen(cf_cmd, stdout=cf_log, stderr=subprocess.STDOUT)
    open('/content/cloudflared.pid', 'w').write(str(cf.pid))
    pat = re.compile(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com')
    for i in range(90):
        time.sleep(1)
        txt = open(cf_log_path, errors='ignore').read()
        m = pat.search(txt)
        if m:
            url = m.group(0).rstrip('/')
            break
        if cf.poll() is not None:
            print(open(cf_log_path, errors='ignore').read()[-2000:])
            log('⚠️ cloudflared chết — Cell 5 (localtunnel)')
            break
    if url:
        open('/content/comfy_url.txt', 'w').write(url)
        try:
            with urllib.request.urlopen(url + '/system_stats', timeout=25) as r:
                log(f'Tunnel HTTP {r.status}')
        except Exception as e:
            log(f'Tunnel HTTP: {e} (403 thường do bấm link trong Colab — hãy DÁN tab mới)')
else:
    log('Không bật tunnel (theo form). ComfyUI chỉ local 127.0.0.1')

print()
print('=' * 64)
print('CẤU HÌNH ĐANG DÙNG')
print(f'  VRAM={VRAM}  PREC={FORCE_PREC}  ATTN={ATTENTION}  PREVIEW={PREVIEW}')
print(f'  VAE={VAE_PREC}  UNET={UNET_PREC}  reserve={RESERVE_VRAM_GB}GB  FAST={FAST}')
print(f'  listen={LISTEN}:{PORT}  CORS={CORS} ({CORS_ORIGIN})  tunnel={TUNNEL}')
print('=' * 64)
if url:
    print()
    print('🎨 COPY LINK, DÁN VÀO THANH ĐỊA CHỈ TAB MỚI (đừng bấm trong Colab):')
    print()
    print('   ' + url)
    print()
elif need_tunnel:
    print('⚠️ Chưa có link Cloudflare → Cell 5')
else:
    print(f'Local: http://127.0.0.1:{int(PORT)}')
print('=' * 64)
print()
print('Trong ComfyUI:')
print('  1. Kéo WAI_ChiTietNho.json')
print('  2. Nhấn R')
print('  3. NẠP MODEL → WAI-illustrious.safetensors')
print('  4. YOLO: bbox/face_yolov8m.pt | bbox/hand_yolov8s.pt | bbox/foot_anime_yolo11m_v3.pt')
print('  5. SAMLoader → sam_vit_b_01ec64.pth')
print('  6. Queue')
print()
print('OOM / đỏ VRAM: chạy lại Cell 3, chọn VRAM=lowvram, PREVIEW=none, tắt FAST.')
print('Ảnh:', OUTPUT_DIR, ' — Save image trước khi ngắt phiên.')
print('Nén: !zip -r /content/anh.zip', OUTPUT_DIR)
print('Cell 4 = health check.  Cell 5 = loca.lt dự phòng.')
print('💡 ComfyUI vẫn chạy nền sau khi cell xong.')


In [ ]:
# ===== CELL 3B (TÙY CHỌN): Giao diện gọn tiếng Việt =====
!pip install -q gradio websocket-client
!wget -q -O /content/giaodien_tao_anh.py https://raw.githubusercontent.com/caone1196-sketch/t-i-li-u/arena/01a09a8b-t-i-li-u/giaodien_tao_anh.py
print('Đợi link https://xxxx.gradio.live — đừng dừng cell khi đang dùng.\n')
!python /content/giaodien_tao_anh.py

In [ ]:
# ===== CELL 4: Kiểm tra =====
!curl -s -o /dev/null -w "A) ComfyUI: HTTP %{http_code}\n" --max-time 20 http://127.0.0.1:8188/system_stats
import re, os
txt = open('/content/cloudflared.log').read() if os.path.exists('/content/cloudflared.log') else ''
m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', txt)
print('Link:', m.group(0) if m else '(chưa có)')
print('\nImpact:'); !ls /content/ComfyUI/custom_nodes | grep -i impact || echo THIEU
print('\nYOLO:'); !ls -lh /content/ComfyUI/models/ultralytics/bbox/ 2>/dev/null || echo THIEU
print('\nSAM:'); !ls -lh /content/ComfyUI/models/sams/ 2>/dev/null || echo THIEU
print('\nlog:'); !tail -30 /content/comfyui.log

In [ ]:
# ===== CELL 5: localtunnel dự phòng =====
!npm install -g localtunnel > /dev/null 2>&1
import urllib.request
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode().strip()
print('🔑 Tunnel Password:', ip)
!lt --port 8188

## 🔧 Cell 1 bị treo — đã sửa gì

Bản trước gộp `pip install ultralytics opencv-python-headless` (im lặng `-q`) → Colab đứng 10–20 phút như chết, đôi khi đè OpenCV/Torch.

Bây giờ:
- Cell 1 chỉ Drive + ComfyUI, **in timestamp từng bước**
- **Không** `force_remount` Drive
- **Không** xóa ComfyUI nếu đã clone
- Cell 1B: Impact + `ultralytics --no-deps` (không cài lại opencv/torch)

**Đứng ở Mounting Drive:** tick `BO_QUA_DRIVE` → Interrupt → chạy lại Cell 1.

**Đứng ở Clone ComfyUI không có %:** mạng GitHub chậm. Interrupt, chạy lại (sẽ resume nếu thư mục lỗi thì xóa `/content/ComfyUI` tay).

Workflow: `WAI_ChiTietNho.json`. Node vẫn đỏ → chưa chạy Cell 1B.